Задача: построить модель бин. классификации, чтобы определить, купит ли пользователь хотя бы что-то за целевой месяц

In [2]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta
from typing import Optional
from pathlib import Path

In [3]:
features_to_use = ["anchor_date", "user_id", 'searches_sum_7d', 'gmv_sum_30d', 'gmv_sum_60d', 'gmv_std_60d', 'target']

In [4]:
FEATURES_PATH = "data/features.parquet"
TARGETS_PATH = "data/targets.parquet"

TRAIN_PATH = "data/train_gen.parquet"
VALID_PATH = "data/valid.parquet"

VALIDATION_DAYS = 30

max_anchor = (
    pl.scan_parquet(FEATURES_PATH)
    .select(pl.col("anchor_date").max())
    .collect()
    .item()
)

validation_start = (
    max_anchor - timedelta(days=VALIDATION_DAYS)
)


dataset = (
    pl.scan_parquet(FEATURES_PATH)
    .join(
        pl.scan_parquet(TARGETS_PATH),
        on=["anchor_date", "user_id"],
        how="left",
    )
)

# TRAIN
(
    dataset
    .filter(
        pl.col("anchor_date") < validation_start
    )
    .select(features_to_use)
    .sink_parquet(
        TRAIN_PATH,
        compression="zstd",
        
    )
)

# VALIDATION
(
    dataset
    .filter(
        pl.col("anchor_date") >= validation_start
    )
    .select(features_to_use)
    .sink_parquet(
        VALID_PATH,
        compression="zstd",
    )
)

In [5]:
tech_cols = ["anchor_date", "user_id"]
target_col = "target"

# all_columns = pl.scan_parquet(TRAIN_PATH).collect_schema().names()
feature_cols = [col for col in features_to_use if col not in tech_cols + [target_col]]

train_lazy = pl.scan_parquet(TRAIN_PATH)
X_train = train_lazy.select(feature_cols).collect()
y_train = train_lazy.select(target_col).collect()


valid_lazy = pl.scan_parquet(VALID_PATH)
X_test = valid_lazy.select(feature_cols).collect()
y_test = valid_lazy.select(target_col).collect()

print(f"Train features shape: {X_train.shape}, Target shape: {y_train.shape}")
print(f"Test features shape: {X_test.shape}, Target shape: {y_test.shape}")


Train features shape: (2000000, 4), Target shape: (2000000, 1)
Test features shape: (500000, 4), Target shape: (500000, 1)


In [6]:
train_lazy.collect()

anchor_date,user_id,searches_sum_7d,gmv_sum_30d,gmv_sum_60d,gmv_std_60d,target
date,i32,f32,f32,f32,f32,f32
2025-11-15,457512,8.0,0.0,18.070642,4.145689,19.406618
2025-11-15,457610,14.0,0.0,6.554982,1.465739,708.317688
2025-11-15,457683,0.0,0.0,0.0,0.0,0.0
2025-11-15,457716,0.0,0.0,0.0,0.0,102.375259
2025-11-15,457745,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…
2025-10-16,457401,0.0,0.0,0.0,0.0,0.0
2025-10-16,457440,13.0,0.0,186.241714,65.84639,15.732052
2025-10-16,457442,1.0,0.0,0.0,0.0,17.177065


In [7]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

tech_cols = ["anchor_date", "user_id"]
target_col = "target"

# Назначаем список признаков (убеждаемся, что в нем нет технических колонок и таргета)
feature_cols = [col for col in features_to_use if col not in tech_cols + [target_col]]

# --- ЗАГРУЗКА ДАННЫХ ИЗ PARQUET ---
# Для временных рядов критически важно добавить .sort("anchor_date") перед .collect()
train_lazy = pl.scan_parquet(TRAIN_PATH).sort("anchor_date")
valid_lazy = pl.scan_parquet(VALID_PATH).sort("anchor_date")

# Собираем матрицы (Выполняем один .collect() на датасет — это экономит память и время)
df_train = train_lazy.collect()
df_test = valid_lazy.collect()

X_train = df_train.select(feature_cols)
y_train = df_train.select(target_col)
y_train = (y_train.to_numpy().ravel() > 0).astype(np.int32)

X_test = df_test.select(feature_cols)
y_test = df_test.select(target_col)
y_test = (y_test.to_numpy().ravel() > 0).astype(np.int32)

print(f"Train features shape: {X_train.shape}, Target shape: {y_train.shape}")
print(f"Test features shape: {X_test.shape}, Target shape: {y_test.shape}")


# --- ИНТЕГРАЦИЯ С CATBOOST ---

# 1. Автоматически находим индексы категориальных колонок для CatBoost
cat_features_idx = [
    i for i, col in enumerate(feature_cols) 
    if df_train[col].dtype in [pl.Categorical, pl.String]
]

# 2. Создаем легковесные и быстрые пулы данных (используем .to_numpy() для CatBoost)
train_pool = Pool(
    data=X_train.to_numpy(), 
    label=y_train.ravel(), 
    cat_features=cat_features_idx
)

valid_pool = Pool(
    data=X_test.to_numpy(), 
    label=y_test.ravel(), 
    cat_features=cat_features_idx
)

# 3. Инициализация и обучение модели
model = CatBoostClassifier(
    iterations=1000,
    eval_metric='AUC',
    loss_function='Logloss',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True, plot=True)

Train features shape: (2000000, 4), Target shape: (2000000,)
Test features shape: (500000, 4), Target shape: (500000,)


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.20714
0:	test: 0.8086515	best: 0.8086515 (0)	total: 264ms	remaining: 4m 23s
100:	test: 0.8158861	best: 0.8158861 (100)	total: 9.54s	remaining: 1m 24s
200:	test: 0.8159309	best: 0.8159325 (195)	total: 18.7s	remaining: 1m 14s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8159356723
bestIteration = 236

Shrink model to first 237 iterations.


CatBoostClassifier(early_stopping_rounds=50, eval_metric='AUC', iterations=1000, loss_function='Logloss', random_seed=42, verbose=100)

In [8]:
import random
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

tech_cols = ["anchor_date", "user_id"]
target_col = "target"

# Формируем чистый список признаков
feature_cols = [col for col in features_to_use if col not in tech_cols + [target_col]]

# --- 1. ЗАГРУЗКА И ДЕПЕРСОНАЛИЗАЦИЯ ДАННЫХ ---
train_lazy = pl.scan_parquet(TRAIN_PATH).sort("anchor_date")
valid_lazy = pl.scan_parquet(VALID_PATH).sort("anchor_date")

df_train = train_lazy.collect()
df_test = valid_lazy.collect()

X_train = df_train.select(feature_cols)
y_train = (df_train.select(target_col).to_numpy().ravel() > 0).astype(np.int32)

X_test = df_test.select(feature_cols)
y_test = (df_test.select(target_col).to_numpy().ravel() > 0).astype(np.int32)

# Индексы категориальных признаков
cat_features_idx = [
    i for i, col in enumerate(feature_cols) 
    if df_train[col].dtype in [pl.Categorical, pl.String]
]

# --- 2. НАСТРОЙКА СЕТКИ ПАРАМЕТРОВ (GRID) ---
# Задайте диапазоны параметров, которые хотите протестировать
param_grid = {
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 5, 8, 12],
    'learning_rate': [0.03, 0.05, 0.1]
}

# Сколько случайных комбинаций проверить (например, 5)
n_iter_search = 5 
random.seed(42)

# Генерируем уникальные комбинации параметров
param_combinations = []
for _ in range(n_iter_search):
    combo = {param: random.choice(values) for param, values in param_grid.items()}
    if combo not in param_combinations:
        param_combinations.append(combo)

# --- 3. ПОДБОР ПАРАМЕТРОВ НА ВРЕМЕННЫХ РЯДАХ (CROSS-VALIDATION) ---
tscv = TimeSeriesSplit(n_splits=3) # 3 фолда для валидации внутри train
best_cv_auc = 0
best_params = None

print(f"Старт подбора параметров (проверяем {len(param_combinations)} комбинаций)...")

for idx, params in enumerate(param_combinations, 1):
    cv_scores = []
    print(f"\n[Кандидат {idx}/{len(param_combinations)}] Тестируем параметры: {params}")
    
    # Кросс-валидация для текущего набора параметров
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train), 1):
        # Быстрый slice в Polars
        df_fold_train = df_train.slice(train_idx[0], len(train_idx))
        df_fold_val = df_train.slice(val_idx[0], len(val_idx))
        
        X_tr = df_fold_train.select(feature_cols).to_numpy()
        y_tr = (df_fold_train.select(target_col).to_numpy().ravel() > 0).astype(np.int32)
        
        X_va = df_fold_val.select(feature_cols).to_numpy()
        y_va = (df_fold_val.select(target_col).to_numpy().ravel() > 0).astype(np.int32)
        
        tr_pool = Pool(X_tr, y_tr, cat_features=cat_features_idx)
        va_pool = Pool(X_va, y_va, cat_features=cat_features_idx)
        
        # Модель для текущего фолда
        fold_model = CatBoostClassifier(
            iterations=600, # Ограничим итерации на этапе поиска для скорости
            eval_metric='AUC',
            loss_function='Logloss',
            early_stopping_rounds=30,
            random_seed=42,
            verbose=False, # Отключаем логи фолдов
            **params
        )
        
        fold_model.fit(tr_pool, eval_set=va_pool, use_best_model=True, plot=True)
        
        # Считаем AUC на внутреннем валидационном фолде
        preds = fold_model.predict_proba(X_va)[:, 1]
        cv_scores.append(roc_auc_score(y_va, preds))
    
    mean_cv_auc = np.mean(cv_scores)
    print(f"-> Средний AUC на кросс-валидационном поиске: {mean_cv_auc:.4f}")
    
    if mean_cv_auc > best_cv_auc:
        best_cv_auc = mean_cv_auc
        best_params = params

print(f"\n Лучшие параметры по результатам CV: {best_params} (CV AUC: {best_cv_auc:.4f})")


# --- 4. ФИНАЛЬНОЕ ОБУЧЕНИЕ И ИТОГОВЫЙ ROC-AUC НА ВАЛИДАЦИОННОМ ДАТАСЕТЕ ---
print("\nОбучение финальной модели на всех тренировочных данных с лучшими параметрами...")

# Создаем полные пулы для финального шага
train_pool = Pool(X_train.to_numpy(), y_train, cat_features=cat_features_idx)
valid_pool = Pool(X_test.to_numpy(), y_test, cat_features=cat_features_idx)

final_model = CatBoostClassifier(
    iterations=1500, # На финальное обучение можно дать больше итераций
    eval_metric='AUC',
    loss_function='Logloss',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
    **best_params # Распаковываем лучшие найденные гиперпараметры
)

# Обучаем с выводом графика (plot=True)
final_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, plot=True)

# Считаем итоговую метрику на отложенном тест-сете (VALID_PATH)
final_preds = final_model.predict_proba(X_test.to_numpy())[:, 1]
final_roc_auc = roc_auc_score(y_test, final_preds)

print(f"\n{'='*50}")
print(f" ИТОГОВЫЙ ROC-AUC НА ОТЛОЖЕННОМ ТЕСТЕ (VALIDATION): {final_roc_auc:.5f}")
print(f"{'='*50}")

Старт подбора параметров (проверяем 5 комбинаций)...

[Кандидат 1/5] Тестируем параметры: {'depth': 8, 'l2_leaf_reg': 1, 'learning_rate': 0.03}
-> Средний AUC на кросс-валидационном поиске: 0.8272

[Кандидат 2/5] Тестируем параметры: {'depth': 8, 'l2_leaf_reg': 8, 'learning_rate': 0.03}
-> Средний AUC на кросс-валидационном поиске: 0.8272

[Кандидат 3/5] Тестируем параметры: {'depth': 4, 'l2_leaf_reg': 5, 'learning_rate': 0.1}
-> Средний AUC на кросс-валидационном поиске: 0.8271

[Кандидат 4/5] Тестируем параметры: {'depth': 4, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
-> Средний AUC на кросс-валидационном поиске: 0.8271

[Кандидат 5/5] Тестируем параметры: {'depth': 6, 'l2_leaf_reg': 1, 'learning_rate': 0.03}
-> Средний AUC на кросс-валидационном поиске: 0.8272

 Лучшие параметры по результатам CV: {'depth': 8, 'l2_leaf_reg': 1, 'learning_rate': 0.03} (CV AUC: 0.8272)

Обучение финальной модели на всех тренировочных данных с лучшими параметрами...


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.8098904	best: 0.8098904 (0)	total: 110ms	remaining: 2m 45s
100:	test: 0.8148862	best: 0.8148862 (100)	total: 12.9s	remaining: 2m 58s
200:	test: 0.8156609	best: 0.8156609 (200)	total: 27.2s	remaining: 2m 55s
300:	test: 0.8158351	best: 0.8158351 (300)	total: 40.9s	remaining: 2m 42s
400:	test: 0.8159165	best: 0.8159165 (400)	total: 54.6s	remaining: 2m 29s
500:	test: 0.8159646	best: 0.8159646 (500)	total: 1m 8s	remaining: 2m 16s
600:	test: 0.8159802	best: 0.8159819 (593)	total: 1m 22s	remaining: 2m 3s
700:	test: 0.8159818	best: 0.8159845 (669)	total: 1m 36s	remaining: 1m 50s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8159844858
bestIteration = 669

Shrink model to first 670 iterations.

 ИТОГОВЫЙ ROC-AUC НА ОТЛОЖЕННОМ ТЕСТЕ (VALIDATION): 0.81598


In [9]:
final_model.save_model("bin_clf.cbm")